In [1]:
import gspread
import pandas as pd
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import os
import pickle

# ==================================================
# PANDAS DISPLAY SETTINGS
# ==================================================
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

# ==================================================
# GOOGLE API SCOPES
# ==================================================
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = None

# ==================================================
# LOAD SAVED TOKEN
# ==================================================
if os.path.exists("token.pkl"):

    with open("token.pkl", "rb") as token:
        creds = pickle.load(token)

# ==================================================
# LOGIN IF TOKEN DOES NOT EXIST
# ==================================================
if not creds or not creds.valid:

    flow = InstalledAppFlow.from_client_secrets_file(
        r"C:\Users\prane\Downloads\client_secret.json",
        SCOPES
    )

    creds = flow.run_local_server(port=0)

    # SAVE TOKEN
    with open("token.pkl", "wb") as token:
        pickle.dump(creds, token)

# ==================================================
# AUTHORIZE GOOGLE SHEETS
# ==================================================
client = gspread.authorize(creds)

print("✅ Logged into YOUR Google Account")

# ==================================================
# GET ALL GOOGLE SHEETS FILES
# ==================================================
all_files = client.list_spreadsheet_files()

print(f"\n✅ Total Sheets Found: {len(all_files)}")

# ==================================================
# LOOP THROUGH ALL SPREADSHEETS
# ==================================================
for file in all_files:

    try:

        print("\n" + "=" * 100)
        print(f"📄 Spreadsheet Name : {file['name']}")
        print("=" * 100)

        # OPEN SPREADSHEET
        spreadsheet = client.open_by_key(file['id'])

        # GET ALL WORKSHEETS
        worksheets = spreadsheet.worksheets()

        print(f"📑 Total Worksheets : {len(worksheets)}")

        # ==================================================
        # LOOP THROUGH WORKSHEETS
        # ==================================================
        for ws in worksheets:

            print("\n" + "-" * 100)
            print(f"📘 Worksheet Name : {ws.title}")
            print("-" * 100)

            # ==================================================
            # GET RAW DATA
            # ==================================================
            data = ws.get_all_values()

            # ==================================================
            # CHECK EMPTY SHEET
            # ==================================================
            if not data:
                print("⚠️ Empty Worksheet")
                continue

            # ==================================================
            # GET HEADER ROW
            # ==================================================
            headers = data[0]

            # ==================================================
            # FIX EMPTY / DUPLICATE HEADERS
            # ==================================================
            fixed_headers = []

            for i, col in enumerate(headers):

                # EMPTY HEADER
                if col.strip() == "":
                    col = f"Column_{i+1}"

                # DUPLICATE HEADER
                original_col = col
                counter = 1

                while col in fixed_headers:
                    col = f"{original_col}_{counter}"
                    counter += 1

                fixed_headers.append(col)

            # ==================================================
            # GET DATA ROWS
            # ==================================================
            rows = data[1:]

            # ==================================================
            # CREATE DATAFRAME
            # ==================================================
            df = pd.DataFrame(rows, columns=fixed_headers)

            # ==================================================
            # CHECK EMPTY DATAFRAME
            # ==================================================
            if df.empty:
                print("⚠️ No Data Found")
                continue

            # ==================================================
            # DISPLAY TABLE
            # ==================================================
            print(df.to_string(index=False))

    except Exception as e:

        print(f"❌ Error in spreadsheet '{file['name']}':")
        print(e)

✅ Logged into YOUR Google Account

✅ Total Sheets Found: 14

📄 Spreadsheet Name : Daily Open roles
📑 Total Worksheets : 1

----------------------------------------------------------------------------------------------------
📘 Worksheet Name : Sheet1
----------------------------------------------------------------------------------------------------
P_ID                        Position                                  Location                 Vendor       End Client  Rate Status                                                                                                                                   Comments
  P0              Power BI Developer Strongsville, OH/ Pittsburgh, PA (Onsite)                                                                                                                                                                                                
 P00    SecDesign Security Architect                                        NY                             

In [2]:
import mysql.connector
import json

# ==================================================
# MYSQL CONNECTION
# ==================================================
db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="YOUR_PASSWORD",
    database="google_sheets_pipeline"
)

cursor = db.cursor()

print("✅ Connected to MySQL")

✅ Connected to MySQL
